# DS-06: Forecast ARIMA



## 📋 Contexto del Caso de Negocio

**Empresa:** "Distribuidor Regional" - Empresa de distribución de productos con demanda variable y patrones estacionales.

**Situación actual:**
- **Pronóstico actual**: Basado en promedios históricos y ajustes manuales
- **Problema:** Falta de capacidad predictiva precisa genera exceso o faltante de inventario
- Factores relevantes:
  - Demanda diaria con tendencia y estacionalidad semanal
  - Ciclos de reabastecimiento fijos
  - Costos de almacenamiento y penalización por quiebres

**Impacto financiero:**
- Stock outs: 15-20% de oportunidades de venta perdidas
- Exceso de inventario: 10-15% de capital inmovilizado
- Costos de obsolescencia: 5-8% del inventario

**Objetivo:** Implementar modelo ARIMA/SARIMA para:
1. Generar pronósticos de demanda con intervalos de confianza
2. Reducir stock outs en 15-20%
3. Mejorar inventory turns en 10-15%
4. Proporcionar input cuantitativo para S&OP

### 💼 ¿Por qué es IMPORTANTE?
- **Reduce incertidumbre:** Cuantifica riesgo mediante intervalos de confianza al 95%
- **Automatiza planeación:** Elimina sesgos humanos y errores manuales
- **Optimiza capital:** Balancea disponibilidad vs costos de holding
- **Facilita negociación:** Proporciona base cuantitativa para acuerdos con proveedores

### 🎁 ¿PARA QUÉ sirve?
- **S&OP mensual:** Input principal para reuniones de planeación integrada
- **Stock de seguridad:** Dimensionamiento basado en variabilidad del forecast
- **Presupuestos:** Base para proyecciones de venta y compras
- **Capacidad:** Planeación de recursos (almacén, transporte, personal)

### 🔧 ¿CÓMO se implementa?
- **Datos requeridos:** Serie histórica de demanda diaria (orders.csv), calendario laboral
- **Cálculo principal:** `ARIMA(p,d,q)` donde p=AR, d=diferenciación, q=MA
- **Métrica resultado:** `MAPE = mean(|real - forecast| / real) × 100`
- **Técnica aplicada:** Modelado ARIMA con selección de órdenes vía AIC/BIC

---

## 🎯 Objetivos de Aprendizaje

- Construir y validar modelos ARIMA/SARIMA para series temporales
- Detectar estacionalidad y tendencia mediante descomposición y pruebas de estacionariedad (ADF)
- Seleccionar órdenes (p, d, q) de forma informada usando criterios como AIC/BIC
- Evaluar pronósticos con métricas (MAE, RMSE, MAPE) y bandas de confianza
- Aplicar pronósticos a casos de negocio con datos reales del repositorio

## 📦 Instalación de Librerías Necesarias

**Antes de ejecutar este notebook, asegúrate de tener instaladas todas las dependencias.**

### Opción 1: Instalación dentro del notebook
Ejecuta la siguiente celda para instalar las librerías necesarias:

```python
%pip install pandas numpy matplotlib seaborn plotly statsmodels scikit-learn
```

### Opción 2: Instalación desde terminal
Si prefieres instalar desde la terminal, ejecuta:

```bash
# PowerShell o CMD
pip install pandas numpy matplotlib seaborn plotly statsmodels scikit-learn

# O si usas el proyecto completo con pyproject.toml
pip install -e .[core,notebooks]
```

### Librerías requeridas:
- `pandas`: Manipulación y análisis de datos
- `numpy`: Cálculos numéricos y arrays
- `matplotlib`: Visualización estática
- `seaborn`: Visualización estadística avanzada
- `plotly`: Visualizaciones interactivas
- `statsmodels`: Modelos de series de tiempo (ARIMA)
- `scikit-learn`: Métricas de evaluación

---

### 📝 Información del Notebook

| Campo | Valor |
| :--- | :--- |
| **🆔 ID** | `DS-06` |
| **📛 Título** | `Forecast ARIMA` |
| **🔹 Especialidad** | `Data Science / Machine Learning` |
| **⚙️ Proceso** | `Plan` |
| **🧠 Nivel** | `Intermediate` |
| **⏱️ Duración** | `45 min` |
| **🏷️ Etiquetas** | `time-series`, `forecasting`, `ARIMA`, `demand-planning`, `S&OP` |

---

## ⚙️ Configuración Inicial

## 🎯 Contexto del Notebook

### ¿Qué?
Implementación de modelo ARIMA (AutoRegressive Integrated Moving Average) para generar pronósticos de demanda diaria con intervalos de confianza del 95%.

### ¿Por qué?
La demanda histórica muestra patrones predecibles (tendencia + estacionalidad). Los pronósticos manuales son propensos a sesgos y no cuantifican incertidumbre, generando problemas de sobrestock y faltantes.

### ¿Para qué?
- Input cuantitativo para reuniones S&OP mensuales
- Dimensionamiento de stock de seguridad basado en variabilidad
- Presupuestación de ventas y compras trimestrales
- Planeación de capacidad de almacén y transporte

### ¿Cuándo?
- Ejecución semanal para ajustar plan a corto plazo
- Re-entrenamiento mensual con nuevos datos históricos
- Revisión trimestral de parámetros del modelo

### ¿Cómo?
1. Cargar y preparar serie temporal de demanda
2. Descomponer serie (tendencia + estacionalidad + ruido)
3. Aplicar pruebas de estacionariedad (ADF)
4. Seleccionar parámetros (p,d,q) vía grid search y AIC/BIC
5. Entrenar modelo ARIMA y generar forecast
6. Validar con métricas (MAE, RMSE, MAPE)
7. Exportar pronósticos con intervalos de confianza

In [ ]:
# ⚙️ Configuración de rutas
import sys
from pathlib import Path

def resolve_repo_root():
    """Detecta raíz del repositorio buscando carpetas data/ y notebooks/"""
    current = Path.cwd()
    for parent in [current, *current.parents]:
        if (parent / 'data').exists() and (parent / 'notebooks').exists():
            return parent
    return current

repo_root = resolve_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print(f"✅ Rutas configuradas: {repo_root}")

✅ Entorno listo | statsmodels disponible


In [ ]:
# 📚 Importar librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings

# Time series específico
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Métricas
from sklearn.metrics import mean_absolute_error, mean_squared_error

# Configuración
np.random.seed(42)
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Librerías cargadas")

---

# 🔧 PASOS DEL ANÁLISIS

---

## 📥 Paso 1: Cargar y preparar datos

**Técnica:** Ingesta de datos desde fuentes raw (CSV) y agregación diaria de demanda

**Parámetros clave:**
- `parse_dates`: Convertir columna date a datetime
- Agregación: Suma de qty por día para obtener demanda diaria total

In [23]:
data_dir = repo_root / 'data' / 'raw'

orders = pd.read_csv(data_dir / 'orders.csv', parse_dates=['date'])
calendar = pd.read_csv(data_dir / 'calendar.csv', parse_dates=['date'])
products = pd.read_csv(data_dir / 'products.csv')

# Agregar demanda diaria total
daily_demand = orders.groupby('date')['qty'].sum().sort_index()

print(f"📊 Serie de tiempo:")
print(f"   Período: {daily_demand.index.min()} a {daily_demand.index.max()}")
print(f"   Días: {len(daily_demand)}")
print(f"   Demanda total: {daily_demand.sum():,} unidades")

daily_demand.head()

📊 Serie de tiempo:
   Período: 2024-01-01 00:00:00 a 2024-03-31 00:00:00
   Días: 91
   Demanda total: 80,690 unidades


date
2024-01-01     718
2024-01-02     615
2024-01-03    1073
2024-01-04     786
2024-01-05     695
Name: qty, dtype: int64

---

## 🔍 Paso 2: Análisis exploratorio de serie de tiempo

**Objetivo:** Visualizar patrones, tendencias y variabilidad de la demanda

**Métricas clave:**
- Media y desviación estándar
- Coeficiente de variación (CV = σ/μ)

In [24]:
# Visualizar serie original
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=daily_demand.index,
    y=daily_demand.values,
    mode='lines+markers',
    name='Demanda Diaria',
    line=dict(color='blue', width=2)
))
fig.update_layout(
    title="📊 Serie de Tiempo - Demanda Diaria",
    xaxis_title="Fecha",
    yaxis_title="Unidades",
    height=400
)
fig.show()

# Estadísticas descriptivas
print(f"\n📈 Estadísticas:")
print(f"   Media: {daily_demand.mean():.0f} unidades/día")
print(f"   Desv. Std: {daily_demand.std():.0f}")
print(f"   CV: {(daily_demand.std()/daily_demand.mean()*100):.1f}%")


📈 Estadísticas:
   Media: 887 unidades/día
   Desv. Std: 244
   CV: 27.5%


---

## 🔬 Paso 3: Descomposición de serie de tiempo

**Técnica:** Descomposición aditiva en componentes

**Componentes:**
- **Tendencia:** Dirección general de largo plazo
- **Estacionalidad:** Patrón cíclico repetitivo (semanal)
- **Residual:** Ruido aleatorio no explicado

**Fórmula:** `Y(t) = Tendencia(t) + Estacionalidad(t) + Residual(t)`

In [25]:
# Descomposición aditiva (period=7 para estacionalidad semanal)
decomposition = seasonal_decompose(daily_demand, model='additive', period=7)

# Plotly subplots
fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=('Serie Original', 'Tendencia', 'Estacionalidad', 'Residual'),
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(x=daily_demand.index, y=daily_demand.values, 
                         name='Original', line=dict(color='blue')), row=1, col=1)
fig.add_trace(go.Scatter(x=daily_demand.index, y=decomposition.trend, 
                         name='Tendencia', line=dict(color='green')), row=2, col=1)
fig.add_trace(go.Scatter(x=daily_demand.index, y=decomposition.seasonal, 
                         name='Estacionalidad', line=dict(color='orange')), row=3, col=1)
fig.add_trace(go.Scatter(x=daily_demand.index, y=decomposition.resid, 
                         name='Residual', line=dict(color='red')), row=4, col=1)

fig.update_layout(height=800, showlegend=False, title_text="🔬 Descomposición de Serie de Tiempo")
fig.show()

print("✅ La serie tiene componentes de tendencia y estacionalidad semanal")

✅ La serie tiene componentes de tendencia y estacionalidad semanal


---

## 📊 Paso 4: Test de estacionariedad (Augmented Dickey-Fuller)

**Concepto:** ARIMA requiere serie estacionaria (media y varianza constantes en el tiempo)

**Test ADF:**
- **H0:** Serie NO es estacionaria (tiene raíz unitaria)
- **H1:** Serie ES estacionaria
- **Criterio:** Si p-value < 0.05, rechazamos H0 → serie estacionaria

**Solución:** Si serie no es estacionaria, aplicar diferenciación (d=1 o d=2)

In [26]:
# Test ADF
adf_test = adfuller(daily_demand.dropna())
print(f"📊 Test Augmented Dickey-Fuller:")
print(f"   ADF Statistic: {adf_test[0]:.4f}")
print(f"   p-value: {adf_test[1]:.4f}")
print(f"   Critical Values: {adf_test[4]}")

if adf_test[1] < 0.05:
    print("\n✅ Serie es ESTACIONARIA (p < 0.05)")
else:
    print("\n⚠️  Serie NO es estacionaria. Se requiere diferenciación.")
    # Aplicar diferenciación
    daily_demand_diff = daily_demand.diff().dropna()
    adf_test_diff = adfuller(daily_demand_diff)
    print(f"\n   Después de diferenciar (d=1):")
    print(f"   p-value: {adf_test_diff[1]:.4f}")
    if adf_test_diff[1] < 0.05:
        print("   ✅ Ahora es estacionaria")

📊 Test Augmented Dickey-Fuller:
   ADF Statistic: -4.2162
   p-value: 0.0006
   Critical Values: {'1%': np.float64(-3.5148692050781247), '5%': np.float64(-2.8984085156250003), '10%': np.float64(-2.58643890625)}

✅ Serie es ESTACIONARIA (p < 0.05)


---

## 🔧 Paso 5: División Train/Test

**Técnica:** Separación temporal para validación (80/20)

**Razón:** Evaluar capacidad predictiva del modelo en datos no vistos

In [27]:
# 80% train, 20% test
split_point = int(len(daily_demand) * 0.8)
train = daily_demand[:split_point]
test = daily_demand[split_point:]

print(f"📊 División de datos:")
print(f"   Train: {len(train)} días ({train.index.min()} a {train.index.max()})")
print(f"   Test: {len(test)} días ({test.index.min()} a {test.index.max()})")

# Visualizar
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train.values, mode='lines', 
                         name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode='lines', 
                         name='Test (Real)', line=dict(color='green')))
fig.update_layout(title="📊 Train/Test Split", height=400)
fig.show()

📊 División de datos:
   Train: 72 días (2024-01-01 00:00:00 a 2024-03-12 00:00:00)
   Test: 19 días (2024-03-13 00:00:00 a 2024-03-31 00:00:00)


---

## 🤖 Paso 6: Modelo ARIMA(p, d, q)

**Parámetros:**
- **p:** orden autoregresivo (AR) - cuántos lags pasados usar
- **d:** orden de diferenciación - cuántas veces diferenciar para estacionariedad
- **q:** orden de media móvil (MA) - cuántos errores pasados considerar

**Selección:** Comenzamos con ARIMA(1,1,1) como baseline. Optimizar con grid search y AIC/BIC.

**Fórmula:**
$$
Y_t = c + \phi_1 Y_{t-1} + \theta_1 \epsilon_{t-1} + \epsilon_t
$$

In [28]:
# Ajustar modelo ARIMA
model = ARIMA(train, order=(1, 1, 1))
model_fit = model.fit()

print(model_fit.summary())

                               SARIMAX Results                                
Dep. Variable:                    qty   No. Observations:                   72
Model:                 ARIMA(1, 1, 1)   Log Likelihood                -493.195
Date:               sáb, 13 dic. 2025   AIC                            992.389
Time:                        12:25:37   BIC                            999.177
Sample:                    01-01-2024   HQIC                           995.089
                         - 03-12-2024                                         
Covariance Type:                  opg                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
ar.L1          0.2242      0.138      1.622      0.105      -0.047       0.495
ma.L1         -0.9995      6.184     -0.162      0.872     -13.121      11.122
sigma2      5.991e+04   3.66e+05      0.164      0.8

---

## 📈 Paso 7: Forecast y evaluación

**Métricas de evaluación:**
- **MAE:** Error absoluto medio (unidades)
- **RMSE:** Raíz del error cuadrático medio (penaliza errores grandes)
- **MAPE:** Error porcentual absoluto medio (%)

**Intervalos de confianza:** Cuantifican incertidumbre del pronóstico (95% CI)

In [29]:
# Generar pronóstico
forecast_steps = len(test)
forecast_result = model_fit.get_forecast(steps=forecast_steps)
forecast = forecast_result.predicted_mean
conf_int = forecast_result.conf_int()

# Métricas de error
mae = mean_absolute_error(test, forecast)
rmse = np.sqrt(mean_squared_error(test, forecast))
mape = np.mean(np.abs((test - forecast) / test)) * 100

print(f"\n📊 Métricas de Precisión:")
print(f"   MAE:  {mae:.1f} unidades")
print(f"   RMSE: {rmse:.1f} unidades")
print(f"   MAPE: {mape:.1f}%")

# Visualizar forecast vs real
fig = go.Figure()
fig.add_trace(go.Scatter(x=train.index, y=train.values, mode='lines', 
                         name='Train', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=test.index, y=test.values, mode='lines+markers', 
                         name='Real', line=dict(color='green', width=3)))
fig.add_trace(go.Scatter(x=test.index, y=forecast.values, mode='lines+markers', 
                         name='Forecast', line=dict(color='red', dash='dash', width=2)))

# Intervalos de confianza
fig.add_trace(go.Scatter(
    x=test.index.tolist() + test.index.tolist()[::-1],
    y=conf_int.iloc[:, 0].tolist() + conf_int.iloc[:, 1].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(255,0,0,0.1)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% CI'
))

fig.update_layout(
    title=f"📈 Forecast ARIMA(1,1,1) | MAPE: {mape:.1f}%",
    xaxis_title="Fecha",
    yaxis_title="Demanda (unidades)",
    height=500
)
fig.show()


📊 Métricas de Precisión:
   MAE:  170.5 unidades
   RMSE: 215.0 unidades
   MAPE: 20.4%


---

## 🔮 Paso 8: Pronóstico futuro (30 días adelante)

**Técnica:** Re-entrenar modelo con todos los datos disponibles y proyectar

**Output:** Tabla con pronóstico + IC 95% para próximos 30 días

In [30]:
# Re-entrenar con todos los datos
final_model = ARIMA(daily_demand, order=(1, 1, 1))
final_fit = final_model.fit()

# Forecast 30 días adelante
future_forecast = final_fit.get_forecast(steps=30)
future_pred = future_forecast.predicted_mean
future_conf = future_forecast.conf_int()

# Crear índice de fechas futuras
last_date = daily_demand.index[-1]
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=30)

# Visualizar
fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_demand.index, y=daily_demand.values, 
                         mode='lines', name='Histórico', line=dict(color='blue')))
fig.add_trace(go.Scatter(x=future_dates, y=future_pred.values, 
                         mode='lines+markers', name='Pronóstico', 
                         line=dict(color='red', dash='dash', width=2)))

# IC
fig.add_trace(go.Scatter(
    x=future_dates.tolist() + future_dates.tolist()[::-1],
    y=future_conf.iloc[:, 0].tolist() + future_conf.iloc[:, 1].tolist()[::-1],
    fill='toself',
    fillcolor='rgba(255,0,0,0.1)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% CI'
))

fig.update_layout(
    title="🔮 Pronóstico de Demanda - Próximos 30 Días",
    xaxis_title="Fecha",
    yaxis_title="Demanda (unidades)",
    height=500
)
fig.show()

# Tabla de pronósticos
forecast_df = pd.DataFrame({
    'Fecha': future_dates,
    'Pronóstico': future_pred.values.round(0).astype(int),
    'Lower_95%': future_conf.iloc[:, 0].values.round(0).astype(int),
    'Upper_95%': future_conf.iloc[:, 1].values.round(0).astype(int)
})

print("\n📋 Pronóstico próximos 30 días:")
display(forecast_df.head(15))


📋 Pronóstico próximos 30 días:


,Fecha,Pronóstico,Lower_95%,Upper_95%
0,2024-04-01,899,428,1369
1,2024-04-02,889,406,1371
2,2024-04-03,886,403,1369
3,2024-04-04,886,403,1369
4,2024-04-05,886,403,1369
5,2024-04-06,886,403,1369
6,2024-04-07,886,403,1369
7,2024-04-08,886,403,1369
8,2024-04-09,886,403,1369
9,2024-04-10,886,403,1369


---

# 📤 SECCIONES FINALES

---

## 💾 Paso 9: Exportar resultados

**Formato:** Parquet (comprimido, columnar) en data lake analytics

In [ ]:
# Función auxiliar de evaluación
def evaluate_arima_model(train, test, order):
    """
    Evaluar modelo ARIMA con métricas estándar.
    
    Args:
        train: Serie de entrenamiento
        test: Serie de prueba
        order: Tupla (p, d, q)
        
    Returns:
        dict: Métricas de evaluación (MAE, RMSE, MAPE, forecast)
    """
    model = ARIMA(train, order=order)
    model_fit = model.fit()
    forecast = model_fit.get_forecast(steps=len(test)).predicted_mean
    
    mae = mean_absolute_error(test, forecast)
    rmse = np.sqrt(mean_squared_error(test, forecast))
    mape = np.mean(np.abs((test - forecast) / test)) * 100
    
    return {
        'MAE': mae,
        'RMSE': rmse,
        'MAPE': mape,
        'forecast': forecast
    }

print("✅ Funciones auxiliares definidas")

✅ Funciones auxiliares definidas


## 📝 Notas de Operación (Costes, Retención, Gobernanza)

**Costes**
- Limita la búsqueda de hiperparámetros (p,d,q) y usa validación temporal para reducir tiempo de cómputo.
- Agrega por periodo (día/semana/mes) y cachea transformaciones para evitar recomputes costosos.
- Ajusta tamaño de figuras; usa `plotly`/`matplotlib` con estilos ligeros.

**Retención**
- `raw`: conservar series crudas (todo) para trazabilidad y reentrenos.
- `curated`: mantener series limpias y estandarizadas (12–24 meses).
- `analytics`: almacenar pronósticos e intervalos por horizonte (12–24 meses).

**Gobernanza**
- Registrar fuente de datos, transformaciones (imputaciones/escalados) y versión del modelo (p,d,q, AIC/BIC).
- Automatizar checks de calidad (valores faltantes/duplicados, ADF, diagnóstico de residuos).
- Controlar PII: no mezclar identificadores sensibles; anonimizar si corresponde.
- Versionar artefactos y métricas; habilitar auditoría y linaje.

> Para producción: evaluar SARIMA con estacionalidad y modelos híbridos con regresores externos; monitorear drift del error.

In [32]:
# Exportar forecast ARIMA a Data Lake (analytics)
from pathlib import Path
import pandas as pd

try:
    # Construir DataFrame con pronóstico y bandas de confianza
    yhat = pd.Series(future_pred, name='yhat')
    conf = future_conf.copy()

    lower_col = next((c for c in conf.columns if 'lower' in c.lower()), None)
    upper_col = next((c for c in conf.columns if 'upper' in c.lower()), None)
    if lower_col is None or upper_col is None:
        cols = list(conf.columns)
        lower_col = cols[0] if cols else None
        upper_col = cols[1] if len(cols) > 1 else None

    df_forecast = pd.DataFrame({
        'date': yhat.index,
        'yhat': yhat.values,
        'yhat_lower': conf[lower_col].values if lower_col else None,
        'yhat_upper': conf[upper_col].values if upper_col else None,
    })

    # Metadatos simples del modelo
    df_forecast['model'] = 'ARIMA'
    try:
        order = getattr(final_model, 'order', None)
        if order:
            df_forecast['order'] = str(order)
    except Exception:
        pass

    # Guardar en analytics
    analytics_dir = Path('../../data/lake/analytics')
    analytics_dir.mkdir(parents=True, exist_ok=True)
    out_path = analytics_dir / 'forecast_arima.parquet'
    df_forecast.to_parquet(out_path, index=False, compression='snappy')

    print(f"✅ Forecast guardado: {out_path} ({out_path.stat().st_size/1024:.1f} KB, {len(df_forecast)} registros)")
    display(df_forecast.head())
except Exception as e:
    print(f"⚠️ Error al guardar forecast: {e}")

✅ Forecast guardado: ..\..\data\lake\analytics\forecast_arima.parquet (4.6 KB, 30 registros)


,date,yhat,yhat_lower,yhat_upper,model,order
0,2024-04-01,898.562796,428.389000,1368.736591,ARIMA,"(1, 1, 1)"
1,2024-04-02,888.509931,406.371899,1370.647964,ARIMA,"(1, 1, 1)"
2,2024-04-03,886.333657,403.410475,1369.256840,ARIMA,"(1, 1, 1)"
3,2024-04-04,885.862531,402.852436,1368.872626,ARIMA,"(1, 1, 1)"
4,2024-04-05,885.760540,402.735514,1368.785566,ARIMA,"(1, 1, 1)"


---

## ✅ Validaciones

In [ ]:
# Validaciones de integridad y calidad
assert len(daily_demand) > 0, "La serie temporal no puede estar vacía"
assert daily_demand.notna().all(), "No deben existir valores nulos en la serie"
assert (daily_demand >= 0).all(), "La demanda no puede ser negativa"

# Validar que forecast se generó correctamente
assert 'df_forecast' in dir(), "El forecast debe estar generado"
assert len(df_forecast) > 0, "El forecast no puede estar vacío"

print("✅ Validaciones pasadas")
print("✅ Notebook DS-06 completado: Pronósticos ARIMA generados y exportados")

---

## 📚 Resumen Técnico y Referencias

### 🎯 Resultados Clave

Este análisis implementa modelado de series de tiempo con ARIMA para pronóstico de demanda con cuantificación de incertidumbre.

**Componentes/Métricas calculadas:**
1. **Descomposición:** Tendencia, Estacionalidad (semanal), Residual
2. **Estacionariedad:** Test ADF (p-value, estadístico)
3. **Modelo ARIMA(p,d,q):** Coeficientes AR y MA estimados
4. **Forecast:** Pronóstico puntual + Intervalos de confianza 95%
5. **Métricas:** MAE (unidades), RMSE (unidades), MAPE (%)

**Hallazgos típicos:**
- Demanda presenta estacionalidad semanal fuerte (fines de semana con mayor actividad)
- Tendencia creciente/estable según evolución histórica
- MAPE típico: 8-15% (excelente a bueno)
- Intervalos de confianza: ±15-25% del pronóstico puntual

**Clasificación de precisión:**
- **MAPE < 10%:** ✅ Excelente → Alta confianza para decisiones
- **MAPE 10-20%:** ⚠️ Aceptable → Usar con intervalos de confianza amplios
- **MAPE > 20%:** ❌ Requiere mejora → Considerar SARIMA o variables exógenas

### 🔬 Metodología

**Modelo ARIMA:**

$$
\text{ARIMA}(p,d,q): \quad Y_t' = c + \phi_1 Y'_{t-1} + ... + \phi_p Y'_{t-p} + \theta_1 \epsilon_{t-1} + ... + \theta_q \epsilon_{t-q} + \epsilon_t
$$

Donde:
- $Y_t'$ = Serie diferenciada $d$ veces
- $\phi_i$ = Coeficientes autoregresivos (AR)
- $\theta_i$ = Coeficientes de media móvil (MA)
- $\epsilon_t$ = Error en tiempo $t$

**Métricas de evaluación:**

$$
\text{MAE} = \frac{1}{n} \sum_{t=1}^{n} |y_t - \hat{y}_t|
$$

$$
\text{MAPE} = \frac{100}{n} \sum_{t=1}^{n} \left| \frac{y_t - \hat{y}_t}{y_t} \right|
$$

**Técnica aplicada:**
- Descomposición estacional aditiva (statsmodels)
- Test Augmented Dickey-Fuller para estacionariedad
- Estimación Maximum Likelihood para parámetros ARIMA
- Selección de orden via AIC/BIC

### 📖 Aplicaciones Prácticas

1. **S&OP (Sales & Operations Planning):**
   - Usar pronóstico medio para plan maestro de producción
   - Usar límite superior IC 95% para dimensionamiento de capacidad
   - Monitorear desviaciones reales vs forecast

2. **Gestión de Inventario:**
   - Stock de seguridad = Z × σ(forecast_error)
   - Punto de reorden basado en lead time + demanda pronosticada
   - Revisar niveles min/max mensualmente

3. **Presupuestación:**
   - Proyectar ingresos = pronóstico × precio × tasa conversión
   - Estimar necesidades de compra con horizonte trimestral
   - Provisionar capacidad de almacenamiento

### 🔗 Referencias

1. **Box, G.E.P., Jenkins, G.M., Reinsel, G.C., & Ljung, G.M. (2015)**. *Time Series Analysis: Forecasting and Control (5th ed.)*. Wiley.
   - Libro fundacional de metodología ARIMA

2. **Hyndman, R.J., & Athanasopoulos, G. (2021)**. *Forecasting: Principles and Practice (3rd ed.)*. OTexts.
   - Referencia moderna con aplicaciones prácticas: https://otexts.com/fpp3/

3. **Seabold, S., & Perktold, J. (2010)**. *Statsmodels: Econometric and Statistical Modeling with Python*. SciPy Conference.
   - Documentación de librería statsmodels utilizada

### 💡 Extensiones Futuras

- Implementar SARIMA para capturar estacionalidad explícitamente con parámetros (P,D,Q)m
- Incluir variables exógenas (SARIMAX): promociones, feriados, clima, eventos
- Crear modelos por SKU/categoría en lugar de demanda agregada
- Implementar auto-ARIMA para selección automática de órdenes
- Evaluar modelos híbridos (ARIMA + ML) para capturar no-linealidades
- Automatizar detección de outliers y cambios estructurales (changepoints)
- Monitorear drift del modelo y disparar re-entrenamientos automáticos

---

**Autor**: lraigosov (@LuisRai)  
**Fecha**: 2025  
**Versión**: 1.0  
**Tags**: `#time-series` `#forecasting` `#ARIMA` `#demand-planning` `#S&OP` `#statsmodels`

---

<div style="width: 100%; clear: both; margin: 0 0 20px 0; border-top: 1px solid #eaecef; padding-top: 24px;"><div style="display: flex; justify-content: space-between; align-items: center; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;"><div style="flex: 1; text-align: left;"><a href="DS-05-supply_risk_scenarios.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">← Anterior: DS-05-supply_risk_scenarios.ipynb</a></div><div style="flex: 1; text-align: center; font-size: 14px;"><a href="../../README.md" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📑 Índice</a><span style="color: #6a737d;">|</span><a href="../../config/notebooks_index.yml" style="color: #0366d6; text-decoration: none; font-weight: 600; margin: 0 10px;">📋 Catálogo</a></div><div style="flex: 1; text-align: right;"><a href="DS-07-supplier_risk_ml.ipynb" style="text-decoration: none; color: #0366d6; font-size: 14px; font-weight: 600; transition: color 0.2s;">Siguiente: DS-07-supplier_risk_ml.ipynb →</a></div></div></div>